# Data Exploration — TriageIQ

Initial EDA on scraped GitHub Issues.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')


In [ ]:
df = pd.read_parquet('../data/processed/issues_microsoft_vscode.parquet')
print(f'Total issues: {len(df)}')
print(f'Date range: {df.created_at.min()} — {df.created_at.max()}')
print(f'State: {df.state.value_counts().to_dict()}')


In [ ]:
# Label distribution
from collections import Counter
all_labels = [l for lbls in df.labels_raw for l in lbls]
top_labels = Counter(all_labels).most_common(30)
print('Top 30 labels:')
for label, count in top_labels:
    print(f'  {label:40s} {count:>6}')


In [ ]:
# Resolution time distribution
closed = df[df.resolution_hours.notna()]
print(f'Closed issues: {len(closed)}')
print(closed.resolution_hours.describe())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(closed.resolution_hours.clip(upper=30*24), bins=50)
ax.set_xlabel('Resolution time (hours, clipped at 30 days)')
ax.set_title('Resolution Time Distribution — microsoft/vscode')
plt.tight_layout()
plt.savefig('../reports/fig_resolution_time_vscode.png', dpi=150)
plt.show()


In [ ]:
# Text length histogram
df['body_len'] = df.body_clean.str.len()
print('Body length stats:')
print(df.body_len.describe())


In [ ]:
# Save summary stats
summary = {
    'total_issues': len(df),
    'date_start': str(df.created_at.min()),
    'date_end': str(df.created_at.max()),
    'closed_count': int(df.state.eq('closed').sum()),
    'open_count': int(df.state.eq('open').sum()),
    'median_resolution_hours': float(closed.resolution_hours.median()),
    'top_labels': top_labels[:10],
}

import json
with open('../reports/01_eda_initial.md', 'w') as f:
    f.write('# Initial EDA — microsoft/vscode\n\n')
    for k, v in summary.items():
        f.write(f'- **{k}**: {v}\n')

print('Saved reports/01_eda_initial.md')


---
## Multi-Repo Analysis

Load all available processed parquets and compare repos side-by-side.

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROCESSED = Path('../data/processed')

# Load all available per-repo parquets (not split files)
repo_dfs = {}
for p in sorted(PROCESSED.glob('issues_*.parquet')):
    key = p.stem.replace('issues_', '')
    repo_dfs[key] = pd.read_parquet(p)

print(f'Loaded {len(repo_dfs)} repos:', list(repo_dfs.keys()))

### Per-repo summary

In [ ]:
rows = []
for repo, rdf in repo_dfs.items():
    closed_mask = rdf['resolution_hours'].notna()
    all_labels = [l for lbls in rdf['labels_raw'] for l in lbls]
    rows.append({
        'repo': repo,
        'total': len(rdf),
        'closed': closed_mask.sum(),
        'open': (~closed_mask).sum(),
        'pct_closed': f"{closed_mask.mean()*100:.1f}%",
        'component_fill': f"{rdf['component'].notna().mean()*100:.1f}%",
        'type_fill': f"{rdf['type'].notna().mean()*100:.1f}%",
        'unique_components': rdf['component'].nunique(),
        'label_vocab': len(set(all_labels)),
        'unique_authors': rdf['author'].nunique(),
        'median_res_days': round(rdf['resolution_hours'].dropna().median() / 24, 1) if closed_mask.sum() > 0 else None,
    })

summary_df = pd.DataFrame(rows).set_index('repo')
summary_df

### Label vocabulary sizes

In [ ]:
fig, axes = plt.subplots(1, len(repo_dfs), figsize=(5 * len(repo_dfs), 5), sharey=False)
if len(repo_dfs) == 1:
    axes = [axes]

for ax, (repo, rdf) in zip(axes, repo_dfs.items()):
    all_labels = [l for lbls in rdf['labels_raw'] for l in lbls]
    top = Counter(all_labels).most_common(20)
    labels_, counts_ = zip(*top) if top else ([], [])
    ax.barh(list(reversed(labels_)), list(reversed(counts_)))
    ax.set_title(repo.replace('_', '/'), fontsize=9)
    ax.set_xlabel('Issue count')

plt.suptitle('Top-20 labels per repo', y=1.02)
plt.tight_layout()
plt.savefig('../reports/fig_label_vocab_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Resolution time distributions (log scale)

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, len(repo_dfs), figsize=(5 * len(repo_dfs), 4), sharey=True)
if len(repo_dfs) == 1:
    axes = [axes]

for ax, (repo, rdf) in zip(axes, repo_dfs.items()):
    res_hours = rdf['resolution_hours'].dropna()
    if len(res_hours) == 0:
        ax.set_title(f'{repo}\n(no closed issues)')
        continue
    log_hours = np.log10(res_hours.clip(lower=0.01))
    ax.hist(log_hours, bins=40, edgecolor='none')
    ax.set_xlabel('log₁₀(resolution hours)')
    ax.set_title(
        f'{repo.replace("_", "/")}\nn={len(res_hours):,}  '
        f'median={res_hours.median()/24:.1f}d  '
        f'p95={res_hours.quantile(0.95)/24:.0f}d',
        fontsize=8,
    )
    for p, v in [(0.5, res_hours.median()), (0.95, res_hours.quantile(0.95))]:
        ax.axvline(np.log10(max(v, 0.01)), linestyle='--', alpha=0.6,
                   label=f'p{int(p*100)}={v/24:.1f}d')
    ax.legend(fontsize=7)

axes[0].set_ylabel('Issue count')
plt.suptitle('Resolution time distributions (log₁₀ hours)', y=1.02)
plt.tight_layout()
plt.savefig('../reports/fig_resolution_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### Cross-repo author overlap

Are any authors active across multiple repos? High overlap → shared domain experts.
Low overlap → siloed communities.

In [ ]:
author_sets = {repo: set(rdf['author'].dropna()) for repo, rdf in repo_dfs.items()}
repos = list(author_sets.keys())

if len(repos) > 1:
    print('Pairwise author overlap (Jaccard similarity):')
    for i, r1 in enumerate(repos):
        for r2 in repos[i+1:]:
            a, b = author_sets[r1], author_sets[r2]
            jaccard = len(a & b) / len(a | b)
            shared = len(a & b)
            print(f'  {r1:30s} ∩ {r2:30s}  |shared|={shared:4d}  Jaccard={jaccard:.3f}')
else:
    print('Only one repo loaded — no cross-repo overlap to compute.')
    repo = repos[0]
    print(f'\nTop-20 most prolific authors in {repo}:')
    print(repo_dfs[repo]['author'].value_counts().head(20).to_string())